In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
import multiprocessing as mp

# 1. Path Resolution
# Add the parent directory to the system path to import modules
# sys.path.append(str(Path(__file__).absolute().parent))
if 'haman' in os.getcwd():
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester')
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester/codes')
    repo_path = Path('/home/haman/mf-temporary')
elif 'pavel' in os.getcwd():
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester')
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester/codes')
    repo_path = Path('/home/pavel/academia/wintermute/mf-temporary')

# 2. The Great MPI Blindfold
keys_to_delete = [k for k in os.environ if k.startswith('SLURM') or k.startswith('OMPI_') or k.startswith('PRTE_') or 'HYDRA' in k]
for key in keys_to_delete:
    del os.environ[key]

# Prevent CPU hogging by the C++ backend
os.environ["OMP_NUM_THREADS"] = "1" 

# 3. Safe Multiprocessing
try:
    mp.set_start_method('spawn', force=True)
    print("Multiprocessing context set to:", mp.get_start_method())
except RuntimeError:
    pass

print(f"Loaded paths securely. MPI blindfolded. Ready for testing!")

In [ ]:
from importlib import reload

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from codes.controller.config import load_workflow_config
from codes.stimuli.loader import load_stimuli_config
from codes.network_params.loader import load_network_parameters

from codes.neuron_simulation import run_neuron_simulation_workflow
from codes.transfer_function import run_tf_fitting_workflow
from codes.mf_simulation import run_mf_simulation_workflow
from codes.snn_simulation import run_snn_simulation_workflow

from codes.plotting import fig_plots
import codes.stimuli as stim
from pydantic import BaseModel

project_path = repo_path / "projects" / "03_stp_models"
os.chdir(project_path)

import pickle


In [ ]:
network_params = load_network_parameters(project_path / "params" / "network_params_new.yaml")
sim_params = load_workflow_config(project_path / "params" / "workflow_params_new.yaml")
stimuli_config = load_stimuli_config(project_path / "params" / "stimuli.yaml")


In [ ]:
# 1. neuron simulation
neuron_results = run_neuron_simulation_workflow(sim_params.neuron_simulation, network_params)

In [ ]:
neuron_results_file = project_path / "results" / "neuron_results.pkl"

if neuron_results_file.exists():
    with open(neuron_results_file, "rb") as f:
        neuron_results = pickle.load(f)
else:
    with open(neuron_results_file, "wb") as f:
        pickle.dump(neuron_results, f)

In [ ]:
fig_name = f"neuron_activity.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_neuron_activity(
    neuron_results, 
    common_params={}, 
    fig_params={
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
        'title': f"Neuron Activity"
    }
)
display(Image(filename=str(fig_path)))

fig_name = f"std_neuron_activity.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_tf_fits_together(
    neuron_results, 
    {"exc_neuron": [],
    "inh_neuron": []
    }, 
    common_params={
        'labels' : [],
        'linestyles' : [],
        # 'xlim' : (0,7),
        'ylim' : (0, 60),
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
        'title': f"Neuron Activity (STD)"
    })
display(Image(filename=str(fig_path)))

In [ ]:
# 2. loop through mf_models and run tf fitting
tf_results_dict = {}
tf_results_legacy = {
    "exc_neuron": [],
    "inh_neuron": []
}
for mf_model_name, mf_sim_params in sim_params.mf_models.items():
    tf_results = run_tf_fitting_workflow(mf_sim_params.transfer_function, network_params, neuron_results)
    tf_results_dict[mf_model_name] = tf_results
    tf_results_legacy["exc_neuron"].append(tf_results["exc_neuron"])
    tf_results_legacy["inh_neuron"].append(tf_results["inh_neuron"])

In [ ]:
fig_name = f"std_neuron_activity.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_tf_fits_together(
    neuron_results,
    tf_results_legacy,
    common_params={
        'labels' : list(sim_params.mf_models.keys()),
        'linestyles' : ["--", "-.", ":"],
        # 'xlim' : (0,7),
        'ylim' : (0, 30),
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
        'title': f"Neuron Activity (STD)"
    })
display(Image(filename=str(fig_path)))

In [ ]:
snn_results_file = project_path / "results" / "snn_results.pkl"

if snn_results_file.exists():
    with open(snn_results_file, "rb") as f:
        snn_results = pickle.load(f)
else:
    snn_results = run_snn_simulation_workflow(sim_params.snn_simulation, network_params, stimuli_config)
    with open(project_path / "results" / "snn_results.pkl", "wb") as f:
        pickle.dump(snn_results, f)

In [ ]:
snn_results = run_snn_simulation_workflow(sim_params.snn_simulation, network_params, stimuli_config)

In [ ]:
snn_results = run_snn_simulation_workflow(sim_params.snn_simulation, network_params, stimuli_config)

mf_results_dict = {}
for mf_model_name, mf_sim_params in sim_params.mf_models.items():
    mf_results = run_mf_simulation_workflow(mf_sim_params, network_params, stimuli_config)
    mf_results_dict[mf_model_name] = mf_results



In [ ]:
for stimulus_name in stimuli_config.keys():
    fig_plots.fig_full_network_overview_together(
        snn_results[stimulus_name], 
        [mf_results_dict[mf_model_name][stimulus_name] for mf_model_name in mf_results_dict.keys()],
        common_params={
            'xmargin': 0.0,
            'ymargin': 0.0,
            'labels': ['SNN'] + list(mf_results_dict.keys()),
            'legend': {'loc': 'upper left'},
            'xlim' : (0, snn_results[stimulus_name].times()[-1])
        },
        fig_params={
            'figsize': (20, 10),  # width, height
            'tight_layout': True,
            'savefig': True,
            'savefig_path': project_path / "imgs" / f"{stimulus_name}_Full_network_overview_together.png",
            'title' : f"Network overview for stimulus: '{stimulus_name}' and drive rate: {stimuli_config[stimulus_name].drive_rate} Hz",
        }
    )

    display(Image(filename=str(project_path / "imgs" / f"{stimulus_name}_Full_network_overview_together.png")))


In [ ]:
for stim_name, result in snn_results.items():
    print(f"{stim_name} - max exc rates {result.exc_rate_mean().max()}")

# snn_results["TwoSidedGaussian"].inh_rate_mean().max()

TODO
- voltage plot for MF Results
- inspection
    - histogram plot of spont activity
    - add overview plots
- runable scripts
- merge two inspection (when I am inspecting more)
- explosion detection within first 1000 ms high activity
    - give it longer time, will the explosion happen again once the adaptation drops?  
    - control of steady state?


- Extractor do not have check on measured variables
- there is no control of units (I have the whole system of working with units, use it)



when inspecting stimulus param I can create stimuli list and go through it and then take the data together

4 stimuli are 2.4 GB data, so keeping it in memory is not wise if there would be finer inspection

maybe best approach is to run one set up, make an overall plot, save the relevant data for inspection (so that I can throw away spikes and unneeded data)

inspection results has to be open for adding additional data

how to save it ["snn"] + mf_names

for spont stimulus I can have time average

for other stimuli I have to make (snn-mf)**2 (or similar measure) and average that over time
error of mf fit compared to snn ground truth. Are there any better measures?




In [ ]:
import codes.controller.inspectors as inspectors
import codes.data_structures.inspection as inspection
from codes.stimuli.config import NoStimulusConfig
import copy

In [ ]:
reload(inspection)

In [ ]:
reload(inspectors)


In [ ]:
test_stimulus = {
  "pattern" : "NoStimulus",
  "stim_params" : {},
  "drive_rate" : 1.5,
  "initial_increase_duration" : 400,
  "simulation_duration" : 2000,
  "stim_target_ratio" : 1.0,
  "target_nodes" : 0,
  "direct_stimulation" : False
}

# Can this be done better? When I want to test a single stimulus, I have to create a StimuliCollection and then access the root of it. This is cumbersome. I want to be able to just pass a dict to the NoStimulusConfig and get a config object back.
# test_stimulus_config = StimuliCollection({"test_stimulus": test_stimulus}).root["test_stimulus"]

test_stim = NoStimulusConfig(**test_stimulus)

inspected_param = "stimulus.drive_rate"
inspected_values = np.array([0.5, 1.0, 1.5])


inspector = inspectors.ParameterInspector(
    base_network_params=network_params,
    base_stimulus_params=test_stim,
    base_sim_params=sim_params,
)


inspection_results = inspector.run_inspection(
    inspected_param=inspected_param,
    inspected_values=inspected_values,
    measured_variables=[
        "exc_rate_time_mean",
        "exc_rate_time_std",
        "inh_rate_time_mean",
        "inh_rate_time_std",
        "exc_adaptation_time_mean",
        "exc_adaptation_time_std",
        "exc_rate_rmse",
        "exc_rate_error_mean",
        "exc_rate_error_std",
        "exc_rate_pearson",
    ],
    start_time=1000.0,
    end_time=2000.0,
)

In [ ]:
inspection_results

In [ ]:
spont_results = inspection_results["spont"]
for measured_variable in spont_results.measured_variables:
    if measured_variable.endswith("_time_std"):
        continue
    fig, ax = plt.subplots(figsize=(8, 6))

    ax.errorbar(
        spont_results.param_values, 
        getattr(spont_results, measured_variable)()[0], 
        yerr=getattr(spont_results, measured_variable.replace("_mean", "_std"))()[0], 
        fmt='o', 
        label = "SNN"
    )
    ax.plot(spont_results.param_values, getattr(spont_results, measured_variable)()[1:].T, label=spont_results.network_names[1:])

    ax.set_title(f"inspected parameter: {spont_results.inspected_param}")
    ax.set_xlabel(f"{spont_results.inspected_param} (Hz)")
    ax.set_ylabel(measured_variable)
    ax.legend()
    fig.tight_layout()
    plt.show()

In [ ]:
dynamic_results = inspection_results["dynamic"]
for measured_variable in dynamic_results.measured_variables:
    fig, ax = plt.subplots(figsize=(8, 6))

    ax.plot(dynamic_results.param_values, getattr(dynamic_results, measured_variable)().T, label=dynamic_results.network_names)

    ax.set_title(f"inspected parameter: {dynamic_results.inspected_param}")
    ax.set_xlabel(f"{dynamic_results.inspected_param} (Hz)")
    ax.set_ylabel(measured_variable)
    ax.legend()
    fig.tight_layout()
    plt.show()

In [ ]:
# from "inspect_divolo-static.ipynb"

fig, axes = plt.subplots(ncols=3, figsize=(16, 8))

plot = ax_plt.FiringRateInspectionPlot({
    "markers": ["o"]+[""]* (len(inspection_results)-1),
    "labels": [result.inspected_network_name for result in inspection_results],
    "linestyles": [""] + [ ':', '--', '--', '--'],
    "legend": True,
    "xlabel": "Drive Rate (Hz)"
})
plot.draw(axes[0], inspection_results)

plot = ax_plt.VoltageInspectionPlot({
    "markers": ["o"]+[""]* (len(inspection_results)-1),
    "labels": [result.inspected_network_name for result in inspection_results],
    "linestyles": [""] + [ ':', '--', '--', '--'],
    "legend": True,
    "xlabel": "Drive Rate (Hz)"
})
plot.draw(axes[1], inspection_results)

plot = ax_plt.AdaptationInspectionPlot({
    "markers": ["o"]+[""]* (len(inspection_results)-1),
    "labels": [result.inspected_network_name for result in inspection_results],
    "linestyles": [""] + [ ':', '--', '--', '--'],
    "legend": True,
    "xlabel": "Drive Rate (Hz)"
})
plot.draw(axes[2], inspection_results)


fig.suptitle("Spontaneous Activity Inspection Results")
fig.tight_layout()
plt.show()

In [ ]:
# Regularity
# Synchrony
# check "inspect_divolo-static.ipynb"

# as measures of AI state, also measures of UP/DOWN states


In [ ]:
# histogram plotting for spont activity stimulus


fig, axes = plt.subplots(1, 5, figsize=(20,5))

# TODO: I have no idea how to properly rescale the histograms!
#   denstity=True does not help, MF underestimates the variance and the normal distrivution is too narrow

# TODO: Derive MF params so that we can fit the voltage as well
# TODO: add MF into adaptation plot, there is only the mean (no std is computed)
# TODO: count is misleading because I do not measure all the neurons, just a subset!! 
#       so it wont sum up to the total number of neurons!!

ax_plt.FiringRateHistogramPlot({
    'binsize': 1,
    'labels' : ['SNN'] + runner.mf_names,
    'start_time': 1000.0,
    'legend': True,
}).draw(axes[0], [snn_results]+mf_results_list)

ax_plt.VoltageHistogramPlot({
    'binsize': 0.4,
    'labels' : ['SNN'],
    'start_time': 1000.0,
    'legend': True,
}).draw(axes[1], [snn_results])

ax_plt.AdaptationHistogramPlot({
    'binsize': 8,
    'labels' : ['SNN'],
    'start_time': 1000.0,
    'legend': True,
}).draw(axes[2], [snn_results])

ax_plt.ExcitatoryNeuronConductanceHistogramPlot({
    'binsize': 0.001,
    'labels' : ['SNN'],
    'start_time': 1000.0,
    'legend': True,
}).draw(axes[3], [snn_results])

ax_plt.InhibitoryNeuronConductanceHistogramPlot({
    'binsize': 0.001,
    'labels' : ['SNN'],
    'start_time': 1000.0,
    'legend': True,
}).draw(axes[4], [snn_results])

fig.suptitle(f"Histograms for stimulus: '{stim_name}' and drive rate: {stimulus['drive_rate']} Hz", fontsize=16)
fig.tight_layout()
plt.show()
